In [1]:
using ArgParse, DataFrames, Statistics
using ArgCheck, JuMP, GLPK, SparseArrays, LinearAlgebra, CSV
using DelimitedFiles
using Plots
using PyPlot
using Ipopt, JSON

In [ ]:
data = "/Users/baperez/Documents/paper/data/cho.sto.csv";

In [ ]:
using DataFrames

sto = CSV.read(data,  DataFrame, delim = ",")
S = sparse(sto[!,1],sto[!,2],sto[!,3])
m, n = Base.size(S)

In [ ]:
mets = CSV.read("/Users/baperez/Documents/paper/data/cho.mets.csv", DataFrame, delim =",")
rxns = CSV.read("/Users/baperez/Documents/paper/data/cho.rxns.csv", DataFrame, delim =",")
@assert Base.size(mets, 1) == m && Base.size(rxns, 1) == n

In [ ]:
e = mets[!,3];
y = mets[!,2];
L = mets[!,6]
V = mets[!,7];
lb = rxns[!,2]
ub = rxns[!,3];
@assert all(lb .≤ 0 .≤ ub)
@assert all(L .≤ 0 .≤ V)

In [ ]:
cDMEMF12 = CSV.read("/Users/baperez/Documents/paper/data/c_DMEM_Advance.txt", DataFrame, delim = ",")
cDMEMlow = CSV.read("/Users/baperez/Documents/paper/data/c_DMEM_Low_Glc.txt", DataFrame, delim = ",")
cDMEMcosto = CSV.read("/Users/baperez/Documents/paper/data/c_fixed_CIMcosto.txt", DataFrame, delim = ",")
C = CSV.read("/Users/baperez/Documents/paper/data/C_gly.txt", DataFrame, delim = ",")
mw = CSV.read("/Users/baperez/Documents/paper/data/mw.txt", DataFrame, delim = ",");# Molar Weigth g/mol
medioOpt01BASAL = readdlm("/Users/baperez/Documents/paper/data/medioOpt01_toxicidad.txt");

In [ ]:
Xdmem = readdlm("/Users/baperez/Documents/paper/data/X05DMEMF12.txt")
Xpremium = readdlm("/Users/baperez/Documents/paper/data/X05Premium.txt")
Xlow = readdlm("/Users/baperez/Documents/paper/data/X05clow.txt");

sdmem = readdlm("/Users/baperez/Documents/paper/data/s05DMEMF12.txt")
spremium = readdlm("/Users/baperez/Documents/paper/data/s05Premium.txt")
slow = readdlm("/Users/baperez/Documents/paper/data/s05clow.txt");

mudmem = readdlm("/Users/baperez/Documents/paper/data/mu05DMEMF12.txt")
mupremium = readdlm("/Users/baperez/Documents/paper/data/mu05Premium.txt")
mulow = readdlm("/Users/baperez/Documents/paper/data/mu05clow.txt");

In [ ]:
w = C[!,2] #costo en el mercado internacional
id = C[!,1] 
mM = mw[!,2] #peso molecular 
@assert all(C[!,2] .≥ 0) #w
@assert all(C[!,3] .≥ 0) #ub
@assert all(C[!,1] == mets[!,1]) #id
@assert all(cDMEM[!,2] .≥ 0) #mM
@assert all(cDMEMcosto[!,2] .≥ 0) #mM

ap = rxns[!,4];
an = rxns[!,5];

In [ ]:
aa = [170,193,197,199,201,202,206,211,213,247,248,249,293,305,312,313,325,331,359,363,365]; #[170,193,197,199,201,202,206,211,213,247,248,249,293,305,312,313,317,325,331,341,342,359,363,365,384,2102,2514,2520];

In [ ]:
DFBA_model = Model()
set_optimizer(DFBA_model,GLPK.Optimizer)

@variable(DFBA_model, u[i=1:m])
@variable(DFBA_model, zeros(n)[i] ≤ rpos[i=1:n] ≤ ub[i])
@variable(DFBA_model, zeros(n)[i] ≤ rneg[i=1:n] ≤ -lb[i])
@variable(DFBA_model, 0 ≤ z ≤ Inf)
@variable(DFBA_model, 0 ≤ a ≤ 1) #costo enzimatico

DFBA_model[:con] = @constraint(DFBA_model,[i=1:m],  L[i] ≤ u[i] ≤ min(V[i],cDMEM[!,2][i]/(0.1*1.0)))  #DFBA_model[:con] = 
@constraint(DFBA_model,[i=1:m], u[i] + sum(S[i,j]*rpos[j] for j=1:n) + sum(-S[i,j]*rneg[j] for j=1:n) - e[i] - y[i]*z == 0) #- p[i]*qp 
@constraint(DFBA_model,sum(ap[j]*rpos[j] + 1.1.*an[j]*rneg[j] for j=1:n) ≤ a) 


@objective(DFBA_model, Max, z)

In [ ]:
# Cambiar las restricciones DFBA_model[:u]
for i = 1:m
   # delete_upper_bound(DFBA_model[:u][i])
    set_upper_bound(DFBA_model[:u][i], min(V[i],cDMEM[!,2][i]/(0.01*1.0)))
end

# Resolver el modelo
optimize!(DFBA_model)
#println(upper_bound.(DFBA_model[:u]))

# Obtener los valores de las variables
value.(z)
#u_values = value.(u)
#rpos_values = value.(rpos)
#rneg_values = value.(rneg)

In [ ]:
function KToxicity(snh4, slac)
    Knh4 = 1.05 #[mM] (efectos de la toxicidad)
    Klac = 8.0 #[mM] (efectos de la toxicidad)
    return 1/((1 + snh4/(Knh4))*(1 + slac/(Klac)))
end

In [ ]:
function DFBA_simulation(X0, s0, t0, dt, tf,death)
    n = Int(ceil((tf - t0) / dt) + 1)
    X = zeros(n)
    mu = zeros(n)
    s = zeros(n, length(cDMEM[!,2]))
    rposlist = zeros(n, length(rxns[!,1]))
    rneglist = zeros(n, length(rxns[!,1]))
    rfinales = zeros(n, length(rxns[!,1]))
    ulist = zeros(n, length(cDMEM[!,2]))
    X[1] = X0
    s[1, :] = s0

    j = 1
    while j < n
        j += 1
        for i = 1:m
            set_upper_bound(DFBA_model[:u][i], min(V[i],s[j-1, :][i]/(X[j-1]*dt + 10^(-10))))
        end
        optimize!(DFBA_model)
        ulist[j,:] = value.(u)
        rposlist[j,:] = value.(rpos)
        rneglist[j,:] = value.(rneg)
        rfinales[j,:] = rposlist[j,:] .- rneglist[j,:]
        mu[j] = value.(z)*KToxicity(s[j,:][319],s[j,:][320])
        X[j] = death*X[j-1] + mu[j] * X[j-1] * dt 
        s[j, :] = s[j-1, :] .- ulist[j,:] * X[j-1] * dt
        
        for k=1:length(cDMEM[!,2])
            if s[j,:][k] < 0 && abs(s[j,:][k]) < 10^(-10)
                s[j,:][k] = abs(s[j,:][k])
            end
        end
        if X[j] < 0
            X[j] = 0
        end
    end
    return X, s, mu, ulist, rfinales
end

In [ ]:
resultsDMEMF12 = DFBA_simulation(0.08,cDMEMF12[!,2],0.1,1,600,1)

In [ ]:
resultsDMEMlow = DFBA_simulation(0.08,cDMEMlow[!,2],0.1,1,600,1)

In [ ]:
resultspremium = DFBA_simulation(0.08,medioOpt01BASAL,0.1,1,600,1)

In [ ]:
Plots.plot(((1.1:1:600.1),resultsDMEMF12[1][2:end]/264/1000*10^6), xscale =:log10,label = "DMEM-F12 Plus",dpi = 300,color=:red,lw=4,xtickfont=font(14), ytickfont=font(14), guidefont=font(14), legendfont=font(10),legend=:topleft,  background_color_legend = nothing)
plot!(((1.1:1:600.1),resultspremium[1][2:end]/264/1000*10^6), xscale =:log10,label = "Optimal medium",lw=4,color=:cyan)
plot!(((1.1:1:600.1),resultsDMEMlow[1][2:end]/264/1000*10^6), xscale =:log10,label = "DMEM Basal",lw=4,color=:gray)
xaxis!("time (hrs)")
yaxis!("Viable Cell Density (10⁶ cells/mL)")
xlims!(1,999)
png("/Users/baperez/Documents/papers/FIGURAS/!NEWS/X.png")

In [ ]:
Plots.plot(((1.1:1:600.1),resultsDMEMF12[3][2:end]), xscale =:log10,label = "DMEM-F12 Plus",dpi = 300,color=:red,lw=4,xtickfont=font(14), ytickfont=font(14), guidefont=font(14), legendfont=font(10),legend=:topright,  background_color_legend = nothing)
plot!(((1.1:1:600.1),resultspremium[3][2:end]), xscale =:log10,label = "Optimal medium",lw=4,color=:cyan)
plot!(((1.1:1:600.1),resultsDMEMlow[3][2:end]), xscale =:log10,label = "DMEM Basal",lw=4,color=:gray)
xaxis!("time (hrs)")
yaxis!("Specific cell growth rate (1/h)")
xlims!(1,999)
png("/Users/baperez/Documents/paper/FIGURAS/!NEWS/mu.png")

In [ ]:
function MincDynamic(X,mu,deltat)
    MincLP = Model()
    set_optimizer(MincLP,GLPK.Optimizer)
    
    @variable(MincLP,zeros(m)[i] ≤ c[i=1:m] ≤ C[!,3][i])
    @variable(MincLP, L[i] ≤ u[i=1:m] ≤ V[i])
    @variable(MincLP, zeros(n)[i] ≤ rpos[i=1:n] ≤ ub[i])
    @variable(MincLP, zeros(n)[i] ≤ rneg[i=1:n] ≤ - lb[i])
    @variable(MincLP, mu ≤ z ≤ Inf)
    @variable(MincLP, 0 ≤ a ≤ 1) #costo enzimatico
      
    @objective(MincLP, Min, sum(w[i]*mw[!,2][i]*c[i]/1000 for i=1:m)) #w[i]*mw[!,2][i]*c[i]/1000
    
    @constraint(MincLP,[i=1:m], u[i] + sum(S[i,j]*rpos[j] for j=1:n) + sum(-S[i,j]*rneg[j] for j=1:n) - e[i] - y[i]*z ==0) # - p[i]*qp
    @constraint(MincLP,[i=1:m], u[i]*X*deltat - c[i] ≤ 0 )
    @constraint(MincLP,sum(ap[j]*rpos[j] + an[j]*rneg[j] for j=1:n) ≤ a) 
    @constraint(MincLP,[i=1:m], c[i] >= cDMEMlow[!,2][i])
    #@constraint(MincLP,[i in [165,166,209,225,231,238,283,286,366,1590]], c[i] == 10000.0)
    #@constraint(MincLP, [i in [213,234,247,249]], c[i] >= medioOpt01BASAL[i])

    optimize!(MincLP)
    
    return value.(c), value.(u), value.(z)
    
end

In [ ]:
medioPruebaDMEM = MincDynamic(sum(Xdmem[i] for i =1:250),mudmem[1],1)

In [ ]:
using Random
using Statistics

In [ ]:
function MincDynamicSensitivity(
        X,mu,deltat,
        w_s,
        L_s,
        V_s,
        e_s,
        y_s,
        ap_s,
        an_s,
        cDMEMlow_s)

    MincLP = Model(GLPK.Optimizer)

    @variable(MincLP,
        zeros(m)[i] ≤ c[i=1:m] ≤ C[!,3][i])

    @variable(MincLP,
        L_s[i] ≤ u[i=1:m] ≤ V_s[i])

    @variable(MincLP,
        zeros(n)[i] ≤ rpos[i=1:n] ≤ ub[i])

    @variable(MincLP,
        zeros(n)[i] ≤ rneg[i=1:n] ≤ -lb[i])

    @variable(MincLP, mu ≤ z ≤ Inf)

    @variable(MincLP, 0 ≤ a ≤ 1)

    @objective(MincLP,
        Min,
        sum(w_s[i]*mw[!,2][i]*c[i]/1000
            for i=1:m))

    @constraint(MincLP,
        [i=1:m],
        u[i] +
        sum(S[i,j]*rpos[j] for j=1:n) +
        sum(-S[i,j]*rneg[j] for j=1:n) -
        e_s[i] -
        y_s[i]*z == 0)

    @constraint(MincLP,
        [i=1:m],
        u[i]*X*deltat - c[i] ≤ 0)

    @constraint(MincLP,
        sum(ap_s[j]*rpos[j] +
            an_s[j]*rneg[j]
            for j=1:n) ≤ a)

    @constraint(MincLP,
        [i=1:m],
        c[i] >= cDMEMlow_s[i])

    optimize!(MincLP)

    return value.(c), value.(u), value(z)

end

In [ ]:
function perturb_nonzero_parameter(
            param;
            nchange=15,
            variation=0.10)

    param_new = copy(param)

    nonzero_idx =
        findall(x -> x != 0, param)

    n = min(nchange,length(nonzero_idx))

    idx =
        shuffle(nonzero_idx)[1:n]

    factors =
        1 .+ (2*variation).*rand(n) .- variation

    param_new[idx] .=
        param[idx] .* factors

    return param_new, idx

end

In [ ]:
Nsim = 100

costs = zeros(Nsim);
growth = zeros(Nsim);

Cmedia = zeros(m,Nsim);

In [ ]:
using Statistics

function run_sensitivity(X,mu,deltat,
            pname;
            Nsim)

    costs = zeros(Nsim)
    growth = zeros(Nsim)
    Cmedia = zeros(m,Nsim)

    for k in 1:Nsim

        w_s = copy(w)
        L_s = copy(L)
        V_s = copy(V)
        e_s = copy(e)
        y_s = copy(y)
        ap_s = copy(ap)
        an_s = copy(an)

        cDMEMlow_s =
            copy(cDMEMlow[!,2])

        if pname == :w
            w_s,_ =
                perturb_nonzero_parameter(w)

        elseif pname == :L
            L_s,_ =
                perturb_nonzero_parameter(L)

        elseif pname == :V
            V_s,_ =
                perturb_nonzero_parameter(V)

        elseif pname == :e
            e_s,_ =
                perturb_nonzero_parameter(e)

        elseif pname == :y
            y_s,_ =
                perturb_nonzero_parameter(y)

        elseif pname == :ap
            ap_s,_ =
                perturb_nonzero_parameter(ap)

        elseif pname == :an
            an_s,_ =
                perturb_nonzero_parameter(an)

        elseif pname == :cDMEMlow
            cDMEMlow_s,_ =
                perturb_nonzero_parameter(
                    cDMEMlow[!,2])
        end

        copt,uopt,zopt =
            MincDynamicSensitivity(
                X,
                mu,
                deltat,
                w_s,
                L_s,
                V_s,
                e_s,
                y_s,
                ap_s,
                an_s,
                cDMEMlow_s)

        costs[k] =
            sum(w_s[i]*
                mw[!,2][i]*
                copt[i]/1000
                for i=1:m)

        growth[k] = zopt
        
        Cmedia[:,k] .= copt

    end

    mean_c = [mean(Cmedia[i,:]) for i in aa]

        std_c = [std(Cmedia[i,:]) for i in aa]

        cv_c = [
            mean_c[j] != 0 ?
            100*std_c[j]/mean_c[j] : 
            NaN
        for j in eachindex(aa)
        ]       

    df_concentration = DataFrame(
    Metabolite = vec(C[!,1][aa]),
    Mean = mean_c,
    Std = std_c,
    CV = cv_c)

    return (

    mean_cost = mean(costs),
    std_cost = std(costs),
    cv_cost = 100*std(costs)/mean(costs),

    mean_growth = mean(growth),
    std_growth = std(growth),
    cv_growth = 100*std(growth)/mean(growth),

    concentration_stats = df_concentration


    )

end

In [ ]:
results = Dict()

for p in [:w,
          :L,
          :V,
          :e,
          :y,
          :ap,
          :an,
          :cDMEMlow]

    results[p] =
        run_sensitivity(sum(Xpremium[i] for i =1:250),mupremium[1],0.5,
                        p,
                        Nsim=100)

end

In [ ]:
ejex=-0.1:6
ejey=ejex
Plots.scatter((results[:e][:concentration_stats][:,2],medioPruebapremium[1][aa]),dpi=300,label = "",color=:red,lw=4,xtickfont=font(12), ytickfont=font(12), guidefont=font(12), legendfont=font(9),legend=:right,)
plot!(ejex,ejey,linestyle=:dash,color=:gray,label = "")
xaxis!("Original concentration (mM)")
yaxis!("Perturbed concentration (mM)")
title!("Sensitivity analysis for parameter e")
png("/Users/baperez/Documents/UH/papers/2do_Journal/Response/2da_Vuelta/Sensibilidad/medio_sensibilidad_e10%.png")